# E2 · BTC/USD (Coinbase) vs BTC/USDT (Binance): volatility and co-movement (exploratory)

The fiat, US-regulated market compared with the stablecoin, offshore market, March 1–21 2023: VAR(1) and Granger
causality on one-minute returns, GARCH(1,1) per market, and a bivariate DCC(1,1) for the time-varying correlation.

> Requires the optional `arch` package: `pip install -e ".[exploratory]"`.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from statsmodels.tsa.api import VAR
from stablecoin_risk import config, volatility_models as vm, plotting
from stablecoin_risk.data import loaders

panel = loaders.load_svb_panel(exchanges=("binance", "coinbase"))
prices = pd.concat({"BTC/USD Coinbase": panel["coinbase_BTCUSD"]["close"],
                    "BTC/USDT Binance": panel["binance_BTCUSDT"]["close"]}, axis=1).dropna()
ret = np.log(prices).diff().dropna()
print(f"{len(ret):,} aligned one-minute returns")

29,962 aligned one-minute returns


In [3]:
periods = {r: ret.loc[a:f"{b} 23:59"] for r, (a, b) in config.REGIMES.items()}
pd.DataFrame({r: {**(x.std() * 1e4).round(2).to_dict(), "correlation": round(x.corr().iloc[0, 1], 3)}
              for r, x in periods.items()}).T.rename_axis("1-min vol (bps) by regime")

,BTC/USD Coinbase,BTC/USDT Binance,correlation
1-min vol (bps) by regime,,,
pre_crisis,5.63,5.54,0.953
peak_crisis,10.95,10.55,0.949
recovery,11.08,11.16,0.939


In [4]:
var = VAR(ret).fit(1)
print(var.test_causality("BTC/USD Coinbase", ["BTC/USDT Binance"], kind="f").summary())
print(var.test_causality("BTC/USDT Binance", ["BTC/USD Coinbase"], kind="f").summary())

Granger causality F-test. H_0: BTC/USDT Binance does not Granger-cause BTC/USD Coinbase. Conclusion: reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
         14.07          3.842   0.000 (1, 59916)
------------------------------------------------
Granger causality F-test. H_0: BTC/USD Coinbase does not Granger-cause BTC/USDT Binance. Conclusion: reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
         504.3          3.842   0.000 (1, 59916)
------------------------------------------------


In [5]:
garch = {c: vm.fit_garch(ret[c]) for c in ret}
pd.DataFrame({c: {"omega": g.params["omega"], "alpha": g.params["alpha[1]"], "beta": g.params["beta[1]"],
                  "alpha+beta": g.params["alpha[1]"] + g.params["beta[1]"], "nu (t dof)": g.params["nu"],
                  "optimizer_flag": g.convergence_flag}
              for c, g in garch.items()}).T.round(4)

,omega,alpha,beta,alpha+beta,nu (t dof),optimizer_flag
BTC/USD Coinbase,0.0914,0.0733,0.9268,1,5.032,8
BTC/USDT Binance,0.1001,0.0671,0.9329,1,4.857,0


In [6]:
z = pd.DataFrame({c: g.std_resid for c, g in garch.items()})
params, rho = vm.fit_dcc(z)
print({k: round(v, 4) if isinstance(v, float) else v for k, v in params.items()})
fig = plotting.plot_svb_panels({"DCC(1,1) correlation, BTC/USD Coinbase vs BTC/USDT Binance": {"DCC correlation": rho}},
                               "correlation", resample="60min", ylim=(0, 1))

{'alpha': 0.0339, 'beta': 0.9596, 'persistence': 0.9935, 'converged': True}


GARCH(1,1)-t: both books are close to integrated (α + β ≈ 1; Coinbase sits on the boundary, hence
optimiser flag 8) with fat tails (ν ≈ 5), so volatility shocks are extremely persistent at one-minute frequency.
Both Granger tests reject, but the lead is asymmetric: Coinbase BTC/USD predicts Binance BTC/USDT far more
strongly (F ≈ 504) than the reverse (F ≈ 14), so the regulated USD market leads price discovery at one-minute
frequency. Volatility doubles in the crisis in *both* books, and their correlation stays around 0.94–0.95
throughout. The dislocation in notebook 02 is therefore specific to the **USDC** quote currency, not a general
break between USD and USDT markets.